# 🐣 Docling을 이용한 문서 인텔리전스: 복잡한 학술 콘텐츠 활용하기

이 노트북에서는 **문서 인텔리전스(Document Intelligence)**를 보여 줍니다. 문서 인텔리전스는 기존 RAG 시스템이 다루기 어려워하는 연구 논문, 학술 자료, 구조화된 콘텐츠 같은 복잡한 문서를 이해하고 처리하는 고급 기능입니다.

**과제:**
연구 논문에서 기본적인 텍스트 추출만으로 교육용 AI 어시스턴트를 만든다고 상상해 보세요. 다음과 같은 정보를 잃게 됩니다.
- 중요한 연구 결과가 담긴 **📊 표 데이터**
- **🧮 수식**과 과학 기호  
- 핵심 인사이트를 제공하는 **📈 차트와 그림**
- 섹션, 참고 문헌, 메타데이터 같은 **🏛️ 문서 구조**
- 학술 논문에서 흔히 쓰이는 **📝 다단 레이아웃**

**해결책: Docling**
Docling은 이러한 복잡한 콘텐츠 구조를 일반 텍스트로 변환하는 고급 문서 처리 도구로, 이미지, PDF, 표 등에서 텍스트를 추출할 수 있습니다.

**만들게 될 것:**
- **🔬 지능형 문서 처리기**: 복잡한 PDF에서 풍부한 콘텐츠 추출
- **📚 향상된 RAG 시스템**: 표, 수식, 구조화된 콘텐츠에 대한 질의  
- **🎯 학술 AI 어시스턴트**: 문서 전체에 대한 이해를 바탕으로 질문에 답변
- **⚡ 프로덕션 파이프라인**: 실제 교육 자료를 대규모로 처리

#### 학술 콘텐츠를 제대로 이해하는 문서 인텔리전스를 만들어 봅시다! 🚀

## 📚 문서 인텔리전스용 라이브러리 가져오기

Docling 처리 기능을 갖춘 문서 인텔리전스 RAG 시스템을 만드는 데 필요한 핵심 라이브러리를 가져옵니다.

In [ ]:
# 핵심 라이브러리
import requests
import json
import os
import sys
sys.path.append('..')

# LLM에 직접 접근하기 위한 OpenAI 호환 클라이언트
from openai import OpenAI

# 벡터 데이터베이스와 임베딩
from pymilvus import connections, utility, Collection, CollectionSchema, FieldSchema, DataType
from sentence_transformers import SentenceTransformer

# 출력용 유틸리티
from termcolor import cprint

## 🔗 Llama 3.2와 Milvus에 연결하기

‼️⚠️ 중요 ⚠️‼️

Milvus 인스턴스에 연결할 수 있도록 아래에 **사용자 이름**과 **클러스터 도메인**을 입력하세요.

In [ ]:
# 중요! 여기에 사용자 이름과 클러스터 도메인을 입력하세요
username = "<USER_NAME>"
cluster_domain = "<CLUSTER_DOMAIN>"

# --- LLM 설정 ---
LLM_ENDPOINT = "http://llama-32-predictor.ai501.svc.cluster.local:8080"
MODEL_NAME = "llama32"

client = OpenAI(
    base_url=LLM_ENDPOINT + "/v1",
    api_key="no-key-required",
)

temperature = 0.0
max_tokens = 512

print(f"✅ LLM endpoint: {LLM_ENDPOINT}")
print(f"   Model: {MODEL_NAME}")

# --- Milvus 설정 ---
collection_name = "docling_rag_collection"
embedding_model_name = "all-MiniLM-L6-v2"
embedding_dim = 384

connections.connect(
    uri=f"http://milvus.{username}-canopy.svc.cluster.local:19530",
    alias="default"
)

print(f"✅ Connected to Milvus at milvus.{username}-canopy.svc.cluster.local:19530")

## 🔬 Docling 처리 함수 구현

이 함수는 Docling 서비스에 연결하고, 앞에서 설명한 지능형 파이프라인으로 문서를 처리합니다.  
복잡한 학술 논문으로 Docling의 문서 인텔리전스를 테스트해 보겠습니다. 표, 수식, 그림이 포함된 실제 연구 논문을 사용합니다 📈

> 참고: Docling은 더 고급 문서 추출 방식을 사용하므로 정보를 추출하는 데 보통 시간이 조금 걸립니다.  
여기서 사용하는 PDF는 서비스 용량에 따라 🕒 3~4분 정도 걸립니다. ☕ 커피 한잔하고 오세요!

In [ ]:
def docling_processing(url):

    # 클러스터에 배포된 Docling 서비스에 연결
    api_address = "http://docling-v0-7-0-predictor.ai501.svc.cluster.local:5001"
    headers = {"Content-Type": "application/json"}

    print(f"🔗 Connecting to Docling service at {api_address}")
    
    # 처리를 위한 Docling 설정
    payload = {
        "http_sources": [{"url": url}],
        "options": {
            "to_formats": ["md"],
            "image_export_mode": "placeholder"
        },
    }
    
    try:
        # 지능형 분석을 위해 문서를 Docling에 전송
        response = requests.post(
            f"{api_address}/v1alpha/convert/source",
            json=payload,
            headers=headers,
            timeout=300
        )
        
        response.raise_for_status()
        
        result_data = response.json()
        md_content = result_data["document"]["md_content"]
        
        return md_content
        
    except requests.exceptions.Timeout:
        print(f"⏰ Processing timeout - complex documents may need more time")
        raise
    except requests.exceptions.RequestException as e:
        print(f"❌ Docling processing failed: {e}")
        raise
    except KeyError as e:
        print(f"❌ Unexpected response format: {e}")
        raise

In [ ]:
# "First Mapping the Canopy Height of Primeval Forests in the Tallest Tree Area of Asia"라는 연구 논문을 사용합니다
url = "https://arxiv.org/pdf/2404.14661"

md_content = docling_processing(url)

print(f"\n🎉 Document intelligence processing complete!")
print(f"📊 Content preview (first 500 characters):")
print(f"{'='*60}")
print(md_content[:500] + "..." if len(md_content) > 500 else md_content)
print(f"{'='*60}")
print(f"📈 Total processed content: {len(md_content)} characters")
print(f"📝 Docling has extracted and structured the complete document content!")

## 📃 처리된 문서를 RAG에 사용하기

이제 Docling으로 처리한 마크다운을 청크로 나누고, 각 청크를 `all-MiniLM-L6-v2`로 임베딩한 뒤, 모두 Milvus 벡터 데이터베이스에 저장합니다. 노트북 2에서 본 것과 같은 데이터베이스입니다.

In [ ]:
# 마크다운을 단어 단위로 겹치는 청크로 나누는 헬퍼
def chunk_text(text, chunk_size=512, overlap=50):
    words = text.split()
    chunks = []
    step = chunk_size - overlap
    for i in range(0, len(words), step):
        chunk = " ".join(words[i : i + chunk_size])
        if chunk:
            chunks.append(chunk)
    return chunks

chunks = chunk_text(md_content)
print(f"📄 Split document into {len(chunks)} chunks")

In [ ]:
# Docling RAG 청크용 Milvus 컬렉션 설정
if utility.has_collection(collection_name):
    utility.drop_collection(collection_name)
    print(f"🧹 Dropped existing collection '{collection_name}'")

id_field = FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=False)
embedding_field = FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=embedding_dim)

schema = CollectionSchema(
    fields=[id_field, embedding_field],
    description="Docling-processed document chunks",
    enable_dynamic_field=False
)

collection = Collection(
    name=collection_name,
    schema=schema,
    using="default",
    shards_num=2,
    consistency_level="Strong"
)

print(f"✅ Created Milvus collection: {collection_name}")

In [ ]:
# 모든 청크를 임베딩하고 Milvus에 삽입
embed_model = SentenceTransformer(embedding_model_name)

print(f"🔄 Embedding {len(chunks)} chunks with '{embedding_model_name}'...")
embeddings = embed_model.encode(chunks, show_progress_bar=True)

# 나중에 검색할 때 쓸 id→텍스트 매핑을 로컬에 보관
id_to_chunk = {i: chunk for i, chunk in enumerate(chunks)}

data = [{"id": i, "embedding": vec.tolist()} for i, vec in enumerate(embeddings)]
collection.insert(data=data)

collection.create_index(
    field_name="embedding",
    index_params={
        "metric_type": "COSINE",
        "index_type": "IVF_FLAT",
        "params": {"nlist": 128}
    },
    index_name="idx"
)

collection.flush()
collection.load()

print(f"\n✅ Document ingestion complete!")
print(f"🎯 {len(chunks)} Docling-processed chunks are now searchable via semantic similarity!")

이제 문서를 수집했으니(벡터 데이터베이스에 추가했으니) 앞에서처럼 질의할 수 있습니다!  
다양한 질의를 자유롭게 시도하며 어떤 답이 나오는지 확인해 보세요.

In [ ]:
# 처리된 문서에 대한 테스트 질의
queries = [
    "What is the PRFXception?",
    "The accuracy values of overall model prediction and residual cross-validation for five regions in southeast Tibet and four regions in northwest Yunnan"
]

for prompt in queries:
    cprint(f"\nUser> {prompt}", "blue")

    # 질의를 임베딩하고 Milvus에서 가장 관련성 높은 청크 검색
    query_embedding = embed_model.encode([prompt])

    results = collection.search(
        data=query_embedding,
        anns_field="embedding",
        param={"metric_type": "COSINE"},
        limit=5,
        output_fields=[]
    )

    cprint(f"\n--- RAG Metadata ---", "yellow")
    retrieved_chunks = []
    for match in results[0]:
        cprint(f"  chunk_id={match.id}  score={match.score:.4f}", "cyan")
        retrieved_chunks.append(id_to_chunk[match.id])

    # 검색된 컨텍스트로 프롬프트 구성
    messages = [{"role": "system", "content": "You are a helpful assistant."}]
    prompt_context = "\n\n".join(retrieved_chunks)
    extended_prompt = (
        f"Please answer the given query using the context below.\n\n"
        f"CONTEXT:\n{prompt_context}\n\nQUERY:\n{prompt}"
    )
    messages.append({"role": "user", "content": extended_prompt})

    # OpenAI 호환 엔드포인트로 Llama 3.2를 직접 호출
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        max_tokens=max_tokens,
        temperature=temperature,
        stream=True,
    )

    cprint("inference> ", color="magenta", end="")
    for chunk in response:
        if chunk.choices and chunk.choices[0].delta.content:
            cprint(chunk.choices[0].delta.content, color="magenta", end="")
    print()  # 스트리밍 출력 후 줄바꿈

In [ ]:
# 정리 - 컬렉션을 메모리에서 해제
collection.release()
# utility.drop_collection(collection_name)  # 컬렉션 삭제

## 🎉 Docling으로 문서 처리를 개선했습니다!

이제 문서 인텔리전스 시스템이 가장 복잡한 학술 콘텐츠도 이해하고 질의할 수 있게 되어, 교육 기관이 지식을 탐색하고 연구하는 방식을 바꿀 수 있습니다! 🚀  
실습 안내 문서로 돌아가 문서 수집을 자동화하는 방법을 알아보세요.